In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler


In [2]:
BASE = Path("../data/processed/primary")

INPUT_FILE = BASE / "carrier_ml_preprocessed.csv"
SCORE_FILE = BASE / "carrier_anomaly_scores.csv"

In [3]:
CHUNK_SIZE = 250_000

# Number of observations used to train the Isolation Forest
TRAIN_SAMPLE_SIZE = 200_000

RANDOM_STATE = 42

In [4]:

header = pd.read_csv(INPUT_FILE, nrows=0)

# Everything in the preprocessed dataset is a feature.
# There are no claim IDs remaining.
FEATURE_COLS = list(header.columns)

print("=" * 80)
print("CARRIER ANOMALY DETECTION")
print("=" * 80)

print("Feature count:", len(FEATURE_COLS))
print("Training sample:", f"{TRAIN_SAMPLE_SIZE:,}")
print("Chunk size:", f"{CHUNK_SIZE:,}")


CARRIER ANOMALY DETECTION
Feature count: 53
Training sample: 200,000
Chunk size: 250,000


In [5]:

# STEP 1 — CREATE TRAINING SAMPLE

print("\nCreating training sample...")

sample_parts = []
sample_rows = 0

for chunk in pd.read_csv(
    INPUT_FILE,
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    remaining = TRAIN_SAMPLE_SIZE - sample_rows

    if remaining <= 0:
        break

    take = min(
        remaining,
        len(chunk)
    )

    sample = chunk.sample(
        n=take,
        random_state=RANDOM_STATE
    )

    sample_parts.append(sample)

    sample_rows += len(sample)

    print(
        f"Collected {sample_rows:,} "
        f"/ {TRAIN_SAMPLE_SIZE:,}"
    )

    if sample_rows >= TRAIN_SAMPLE_SIZE:
        break

train_df = pd.concat(
    sample_parts,
    ignore_index=True
)

del sample_parts

print("\nTraining sample shape:")
print(train_df.shape)



Creating training sample...
Collected 200,000 / 200,000

Training sample shape:
(200000, 53)


In [6]:

# STEP 2 — ROBUST SCALING

print("\nScaling features...")

scaler = RobustScaler()

X_train = scaler.fit_transform(
    train_df[FEATURE_COLS]
)

X_train = np.asarray(
    X_train,
    dtype=np.float32
)

del train_df

print("Scaled training matrix:")
print(X_train.shape)



Scaling features...
Scaled training matrix:
(200000, 53)


In [7]:

# STEP 3 — TRAIN ISOLATION FOREST

print("\nTraining Isolation Forest...")

model = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train)

del X_train

print("Model training complete.")



Training Isolation Forest...
Model training complete.


In [8]:

# STEP 4 — SCORE ALL CARRIER CLAIMS

print("\nScoring all Carrier claims...")

if SCORE_FILE.exists():
    SCORE_FILE.unlink()

first_chunk = True
total_scored = 0

for i, chunk in enumerate(
    pd.read_csv(
        INPUT_FILE,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    X = chunk[FEATURE_COLS]

    X_scaled = scaler.transform(X)

    X_scaled = np.asarray(
        X_scaled,
        dtype=np.float32
    )

    # IsolationForest decision function:
    # higher = more normal
    decision = model.decision_function(
        X_scaled
    )

    # Convert so that:
    # higher = more anomalous
    anomaly_score = -decision

    result = pd.DataFrame({
        "ANOMALY_SCORE": anomaly_score
    })

    result.to_csv(
        SCORE_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    total_scored += len(chunk)

    del X_scaled
    del X
    del result

    print(
        f"Scored {total_scored:,} claims..."
    )

print("\n" + "=" * 80)
print("CARRIER MODEL COMPLETE")
print("=" * 80)

print(
    "Claims scored:",
    f"{total_scored:,}"
)

print(
    "Expected:",
    f"{4_741_335:,}"
)

print(
    "Score file:",
    SCORE_FILE
)


Scoring all Carrier claims...
Scored 250,000 claims...
Scored 500,000 claims...
Scored 750,000 claims...
Scored 1,000,000 claims...
Scored 1,250,000 claims...
Scored 1,500,000 claims...
Scored 1,750,000 claims...
Scored 2,000,000 claims...
Scored 2,250,000 claims...
Scored 2,500,000 claims...
Scored 2,750,000 claims...
Scored 3,000,000 claims...
Scored 3,250,000 claims...
Scored 3,500,000 claims...
Scored 3,750,000 claims...
Scored 4,000,000 claims...
Scored 4,250,000 claims...
Scored 4,500,000 claims...
Scored 4,741,335 claims...

CARRIER MODEL COMPLETE
Claims scored: 4,741,335
Expected: 4,741,335
Score file: ..\data\processed\primary\carrier_anomaly_scores.csv


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

SCORE_FILE = BASE / "carrier_anomaly_scores.csv"
DATA_FILE = BASE / "carrier_ml_ready.csv"

print("=" * 90)
print("CARRIER ANOMALY SCORE ANALYSIS")
print("=" * 90)

# ============================================================
# LOAD SCORES
# ============================================================

scores = pd.read_csv(SCORE_FILE)

print("\nScore rows:", f"{len(scores):,}")
print("Score columns:", list(scores.columns))

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("\nMissing scores:", scores["ANOMALY_SCORE"].isna().sum())

print(
    "Infinite scores:",
    np.isinf(scores["ANOMALY_SCORE"]).sum()
)

print(
    "Unique scores:",
    scores["ANOMALY_SCORE"].nunique()
)

# ============================================================
# SCORE DISTRIBUTION
# ============================================================

print("\n" + "=" * 90)
print("ANOMALY SCORE DISTRIBUTION")
print("=" * 90)

print(
    scores["ANOMALY_SCORE"].describe(
        percentiles=[
            0.001,
            0.005,
            0.01,
            0.02,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

# ============================================================
# TOP ANOMALIES
# ============================================================

print("\n" + "=" * 90)
print("TOP 100 ANOMALOUS CLAIMS")
print("=" * 90)

top_scores = (
    scores
    .sort_values(
        "ANOMALY_SCORE",
        ascending=False
    )
    .head(100)
    .copy()
)

print(top_scores.head(20).to_string(index=False))

# ============================================================
# SCORE THRESHOLDS
# ============================================================

print("\n" + "=" * 90)
print("HIGH ANOMALY THRESHOLDS")
print("=" * 90)

for percentile in [
    90,
    95,
    97,
    98,
    99,
    99.5,
    99.9
]:

    threshold = np.percentile(
        scores["ANOMALY_SCORE"],
        percentile
    )

    count = (
        scores["ANOMALY_SCORE"] >= threshold
    ).sum()

    print(
        f"Top {100-percentile:.1f}% | "
        f"threshold={threshold:.6f} | "
        f"claims={count:,}"
    )

# ============================================================
# NEGATIVE / POSITIVE SCORE CHECK
# ============================================================

print("\n" + "=" * 90)
print("SCORE RANGE")
print("=" * 90)

print(
    "Minimum:",
    scores["ANOMALY_SCORE"].min()
)

print(
    "Maximum:",
    scores["ANOMALY_SCORE"].max()
)

print(
    "Mean:",
    scores["ANOMALY_SCORE"].mean()
)

print(
    "Median:",
    scores["ANOMALY_SCORE"].median()
)

CARRIER ANOMALY SCORE ANALYSIS

Score rows: 4,741,335
Score columns: ['ANOMALY_SCORE']

Missing scores: 0
Infinite scores: 0
Unique scores: 4741169

ANOMALY SCORE DISTRIBUTION
count    4.741335e+06
mean    -5.505274e-02
std      3.801930e-02
min     -1.329446e-01
0.1%    -1.213437e-01
0.5%    -1.160729e-01
1%      -1.131169e-01
2%      -1.094509e-01
5%      -1.030837e-01
10%     -9.639646e-02
25%     -8.235841e-02
50%     -6.181091e-02
75%     -3.594785e-02
90%     -5.578849e-03
95%      1.815165e-02
99%      6.916355e-02
99.5%    8.609984e-02
99.9%    1.180928e-01
max      2.282328e-01
Name: ANOMALY_SCORE, dtype: float64

TOP 100 ANOMALOUS CLAIMS
 ANOMALY_SCORE
      0.228233
      0.210421
      0.209330
      0.203540
      0.201600
      0.200286
      0.197136
      0.196458
      0.195606
      0.195554
      0.195022
      0.194513
      0.193835
      0.193523
      0.192643
      0.191798
      0.191671
      0.191218
      0.190726
      0.190537

HIGH ANOMALY THRESHOLDS
Top 

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

SCORE_FILE = BASE / "carrier_anomaly_scores.csv"
DATA_FILE = BASE / "Carrier_Claims_Features.csv"
OUTPUT_FILE = BASE / "carrier_top_anomalies.csv"

TOP_N = 100
CHUNK_SIZE = 250_000

# ============================================================
# LOAD SCORES
# ============================================================

scores = pd.read_csv(SCORE_FILE)

scores["ROW_ID"] = np.arange(len(scores))

top_rows = scores.nlargest(
    TOP_N,
    "ANOMALY_SCORE"
)[["ROW_ID", "ANOMALY_SCORE"]]

top_indices = set(top_rows["ROW_ID"])

print("=" * 80)
print("RETRIEVING TOP CARRIER ANOMALIES")
print("=" * 80)

print("Scores:", f"{len(scores):,}")
print("Top anomalies:", TOP_N)

# ============================================================
# COLUMNS WE WANT TO INSPECT
# ============================================================

DETAIL_COLS = [
    "CLM_ID",
    "DESYNPUF_ID",
    "total_claim_payment_amt",
    "total_allowed_charge_amt",
    "total_deductible_amt",
    "total_coinsurance_amt",
    "total_primary_payer_paid_amt",
    "avg_payment_per_line",
    "payment_to_allowed_ratio",
    "line_count",
    "unique_hcpcs_count",
    "max_line_payment",
    "diagnosis_count",
    "unique_diagnosis_count",
    "primary_provider_npi",
    "distinct_provider_count_on_claim",
    "provider_claim_volume",
    "provider_avg_claim_payment",
    "claim_year",
    "claim_month",
    "claim_day_of_week"
]

# ============================================================
# STREAM ORIGINAL CARRIER FEATURES
# ============================================================

matches = []
current_row = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_FILE,
        usecols=DETAIL_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    start = current_row
    end = current_row + len(chunk)

    relevant = [
        idx for idx in top_indices
        if start <= idx < end
    ]

    if relevant:

        local_indices = [
            idx - start
            for idx in relevant
        ]

        selected = chunk.iloc[
            local_indices
        ].copy()

        selected["ROW_ID"] = relevant

        matches.append(selected)

    current_row = end

    print(
        f"Chunk {chunk_number}: "
        f"{current_row:,} rows processed"
    )

# ============================================================
# COMBINE TOP CLAIMS
# ============================================================

claims = pd.concat(
    matches,
    ignore_index=True
)

claims = claims.merge(
    top_rows,
    on="ROW_ID",
    how="left"
)

claims = claims.sort_values(
    "ANOMALY_SCORE",
    ascending=False
).reset_index(drop=True)

claims["ANOMALY_RANK"] = (
    np.arange(len(claims)) + 1
)

# Put important columns first

first_cols = [
    "ANOMALY_RANK",
    "ANOMALY_SCORE",
    "CLM_ID",
    "DESYNPUF_ID",
    "primary_provider_npi"
]

remaining_cols = [
    c for c in claims.columns
    if c not in first_cols + ["ROW_ID"]
]

claims = claims[
    first_cols + remaining_cols
]

# ============================================================
# SAVE
# ============================================================

claims.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 80)
print("TOP CARRIER ANOMALIES SAVED")
print("=" * 80)

print("Rows:", len(claims))
print("Saved:", OUTPUT_FILE)

print("\nTOP 20:")
print(
    claims.head(20).to_string(
        index=False
    )
)

RETRIEVING TOP CARRIER ANOMALIES
Scores: 4,741,335
Top anomalies: 100
Chunk 1: 250,000 rows processed
Chunk 2: 500,000 rows processed
Chunk 3: 750,000 rows processed
Chunk 4: 1,000,000 rows processed
Chunk 5: 1,250,000 rows processed
Chunk 6: 1,500,000 rows processed
Chunk 7: 1,750,000 rows processed
Chunk 8: 2,000,000 rows processed
Chunk 9: 2,250,000 rows processed
Chunk 10: 2,500,000 rows processed
Chunk 11: 2,750,000 rows processed
Chunk 12: 3,000,000 rows processed
Chunk 13: 3,250,000 rows processed
Chunk 14: 3,500,000 rows processed
Chunk 15: 3,750,000 rows processed
Chunk 16: 4,000,000 rows processed
Chunk 17: 4,250,000 rows processed
Chunk 18: 4,500,000 rows processed
Chunk 19: 4,741,335 rows processed

TOP CARRIER ANOMALIES SAVED
Rows: 100
Saved: ..\data\processed\primary\carrier_top_anomalies.csv

TOP 20:
 ANOMALY_RANK  ANOMALY_SCORE          CLM_ID      DESYNPUF_ID primary_provider_npi  total_claim_payment_amt  total_allowed_charge_amt  total_deductible_amt  total_coinsuranc

In [11]:
import pandas as pd
from pathlib import Path

BASE = Path("../data/processed/primary")

FILES = [
    BASE / "Carrier_Claims_Features.csv",
    BASE / "Carrier_Claims_Combined_FULL.csv",
    BASE / "carrier_claims_with_beneficiary_features.csv",
    BASE / "carrier_ml_ready.csv",
]

wanted = [
    "CLM_ID",
    "DESYNPUF_ID",
    "primary_provider_npi",
    "total_claim_payment_amt",
    "total_allowed_charge_amt",
    "payment_to_allowed_ratio",
]

print("=" * 80)
print("CARRIER SOURCE COLUMN CHECK")
print("=" * 80)

for file in FILES:

    if not file.exists():
        print("\nNOT FOUND:", file)
        continue

    header = pd.read_csv(file, nrows=0)

    print("\nFILE:", file.name)
    print("Columns:", len(header.columns))

    found = [
        col for col in wanted
        if col in header.columns
    ]

    missing = [
        col for col in wanted
        if col not in header.columns
    ]

    print("Found:", found)
    print("Missing:", missing)

CARRIER SOURCE COLUMN CHECK

FILE: Carrier_Claims_Features.csv
Columns: 21
Found: ['CLM_ID', 'DESYNPUF_ID', 'primary_provider_npi', 'total_claim_payment_amt', 'total_allowed_charge_amt', 'payment_to_allowed_ratio']
Missing: []

FILE: Carrier_Claims_Combined_FULL.csv
Columns: 142
Found: ['CLM_ID', 'DESYNPUF_ID']
Missing: ['primary_provider_npi', 'total_claim_payment_amt', 'total_allowed_charge_amt', 'payment_to_allowed_ratio']

FILE: carrier_claims_with_beneficiary_features.csv
Columns: 52
Found: ['CLM_ID', 'DESYNPUF_ID', 'primary_provider_npi', 'total_claim_payment_amt', 'total_allowed_charge_amt', 'payment_to_allowed_ratio']
Missing: []

FILE: carrier_ml_ready.csv
Columns: 55
Found: ['CLM_ID', 'DESYNPUF_ID', 'total_claim_payment_amt', 'total_allowed_charge_amt', 'payment_to_allowed_ratio']
Missing: ['primary_provider_npi']


In [13]:
print("=" * 80)
print("CHECKING TRAINED ANOMALY MODELS")
print("=" * 80)

for name in [
    "carrier_model",
    "outpatient_model",
    "inpatient_model",
    "carrier_if",
    "outpatient_if",
    "inpatient_if",
    "carrier_scaler",
    "outpatient_scaler",
    "inpatient_scaler",
    "scaler",
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

print("=" * 80)

CHECKING TRAINED ANOMALY MODELS
scaler: RobustScaler


In [14]:
from sklearn.ensemble import IsolationForest

print("=" * 80)
print("TRAINED ISOLATION FOREST OBJECTS")
print("=" * 80)

for name, obj in globals().items():
    try:
        if isinstance(obj, IsolationForest):
            print(f"{name} -> IsolationForest")
    except:
        pass

print("\n" + "=" * 80)
print("SCALERS")
print("=" * 80)

from sklearn.preprocessing import RobustScaler, StandardScaler

for name, obj in globals().items():
    try:
        if isinstance(obj, (RobustScaler, StandardScaler)):
            print(f"{name} -> {type(obj).__name__}")
    except:
        pass

TRAINED ISOLATION FOREST OBJECTS
model -> IsolationForest

SCALERS
scaler -> RobustScaler


In [15]:
import joblib
import json
from pathlib import Path

# ============================================================
# SAVE CARRIER ANOMALY MODEL
# ============================================================

MODEL_DIR = Path("../models/anomaly/carrier")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save trained model
joblib.dump(
    model,
    MODEL_DIR / "isolation_forest.joblib"
)

# Save scaler
joblib.dump(
    scaler,
    MODEL_DIR / "scaler.joblib"
)

# ------------------------------------------------------------
# IMPORTANT:
# Replace this with the EXACT FEATURE_COLS used when training
# the Carrier model.
# ------------------------------------------------------------

feature_schema = {
    "claim_type": "CARRIER",
    "model_type": "IsolationForest",
    "scaler_type": type(scaler).__name__,
    "feature_count": len(FEATURE_COLS),
    "features": list(FEATURE_COLS)
}

with open(
    MODEL_DIR / "feature_schema.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(feature_schema, f, indent=2)

print("=" * 80)
print("CARRIER MODEL SAVED")
print("=" * 80)

print("Model:", MODEL_DIR / "isolation_forest.joblib")
print("Scaler:", MODEL_DIR / "scaler.joblib")
print("Schema:", MODEL_DIR / "feature_schema.json")
print("Features:", len(FEATURE_COLS))
print("Scaler type:", type(scaler).__name__)

CARRIER MODEL SAVED
Model: ..\models\anomaly\carrier\isolation_forest.joblib
Scaler: ..\models\anomaly\carrier\scaler.joblib
Schema: ..\models\anomaly\carrier\feature_schema.json
Features: 53
Scaler type: RobustScaler
